# 03 - Banco de dados: normalização, SQLite e Mongita

Notebook da etapa 3: perfil dos JSONL, decisões de normalização, criação e validação de `banco/hotel.db`, consultas de negócio em SQL e armazenamento documental complementar em Mongita.

## 1. Inspeção dos dados

In [4]:
from pathlib import Path
import json, sqlite3, shutil
import pandas as pd
from mongita import MongitaClientDisk
from IPython.display import display

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "dados").is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Não encontrei a pasta 'dados' neste diretório ou em seus diretórios ancestrais.")
DATA = ROOT / "dados"
BANK = ROOT / "banco"
BANK.mkdir(exist_ok=True)
DB = BANK / "hotel.db"
entities = {"hotels.jsonl":"HOTEL", "room_types.jsonl":"ROOM_TYPE", "reviews.jsonl":"Documento REVIEWS (Mongita)", "review_scenarios.jsonl":"REVIEW_SCENARIO", "rate_decisions.jsonl":"RATE_DECISION", "daily_market.jsonl":"DAILY_MARKET", "weekly_performance.jsonl":"WEEKLY_PERFORMANCE"}
def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]
data = {name: read_jsonl(DATA / name) for name in entities}
profile=[]
for filename, rows in data.items():
    df=pd.DataFrame(rows)
    profile.append({"arquivo":filename,"entidade":entities[filename],"linhas":len(rows),"colunas":", ".join(df.columns),"tipos observados":"; ".join(f"{c}: {t}" for c,t in df.dtypes.items()),"nulos/vazios":"; ".join(f"{c}: {int(df[c].isna().sum()+df[c].map(lambda v:v=='').sum())}" for c in df if df[c].isna().any() or df[c].map(lambda v:v=='').any()) or "nenhum"})
display(pd.DataFrame(profile))
unique_fields={"hotels.jsonl":["tier","brand","city","comp_set"],"room_types.jsonl":["room_type","view","tier"],"reviews.jsonl":["scenario_id","rating","channel","segment","sentiment"],"review_scenarios.jsonl":["scenario_id","expected_sentiment"],"rate_decisions.jsonl":["action","outcome","demand_event"],"daily_market.jsonl":["season","rate_action","demand_event"],"weekly_performance.jsonl":["demand_event"]}
display(pd.DataFrame([{"arquivo":f,"campo":k,"valores_unicos":len({str(r.get(k)) for r in data[f] if r.get(k) not in (None,"")}),"valores/exemplos":", ".join(sorted({str(r.get(k)) for r in data[f] if r.get(k) not in (None,"")})[:20])} for f,ks in unique_fields.items() for k in ks]))

,arquivo,entidade,linhas,colunas,tipos observados,nulos/vazios
0,hotels.jsonl,HOTEL,14,"hotel_id, name, brand, tier, rooms, opened_yea...",hotel_id: str; name: str; brand: str; tier: st...,nenhum
1,room_types.jsonl,ROOM_TYPE,42,"hotel_id, room_type_id, room_type, rooms, max_...",hotel_id: str; room_type_id: str; room_type: s...,nenhum
2,reviews.jsonl,Documento REVIEWS (Mongita),12077,"review_id, hotel_id, review_date, stay_date, w...",review_id: str; hotel_id: str; review_date: st...,response_text: 3769; response_hours: 3769
3,review_scenarios.jsonl,REVIEW_SCENARIO,11,"scenario_id, label, share_of_reviews, rating_b...",scenario_id: str; label: str; share_of_reviews...,nenhum
4,rate_decisions.jsonl,RATE_DECISION,1442,"decision_id, hotel_id, week_start, action, cur...",decision_id: str; hotel_id: str; week_start: s...,demand_event: 1246
5,daily_market.jsonl,DAILY_MARKET,10192,"hotel_id, hotel_name, tier, comp_set, date, we...",hotel_id: str; hotel_name: str; tier: str; com...,demand_event: 9534
6,weekly_performance.jsonl,WEEKLY_PERFORMANCE,1456,"hotel_id, hotel_name, tier, comp_set, week_sta...",hotel_id: str; hotel_name: str; tier: str; com...,demand_event: 1246; flow_through: 728; mean_ra...


,arquivo,campo,valores_unicos,valores/exemplos
0,hotels.jsonl,tier,5,"boutique, economy, luxury, midscale, upscale"
1,hotels.jsonl,brand,8,"Grand Hotels, Harborlight, Independent, Kestre..."
2,hotels.jsonl,city,1,Port Alder
3,hotels.jsonl,comp_set,5,"boutique, economy, luxury, midscale, upscale"
4,room_types.jsonl,room_type,15,"Accessible Room, Bay View King, Corner Suite, ..."
5,room_types.jsonl,view,5,"city, courtyard, harbour, park, street"
6,room_types.jsonl,tier,5,"boutique, economy, luxury, midscale, upscale"
7,reviews.jsonl,scenario_id,11,"billing, breakfast, checkin_wait, cleanliness,..."
8,reviews.jsonl,rating,5,"1, 2, 3, 4, 5"
9,reviews.jsonl,channel,4,"OTA survey, direct survey, review site, social"


## 2. Análise da normalização

A modelagem aproxima-se da 3FN para atributos descritivos. A tabela **HOTEL** mantém os atributos dependentes de `hotel_id`; campos como `hotel_name`, `tier` e `comp_set` redundantes são removidos das tabelas de fatos e recuperados por `JOIN`. `ROOM_TYPE` é dependente do hotel. A lista de comodidades é separada em **AMENITY** e na associativa **HOTEL_AMENITY**. `REVIEW_SCENARIO` guarda uma vez o catálogo de cenários, enquanto **REVIEW** guarda os fatos observados de avaliação e referencia hotel e cenário.

**Correção aplicada:** a versão anterior mantinha as avaliações apenas em Mongita e copiava temporariamente os documentos para um SQLite em memória. Isso não atendia integralmente o modelo relacional solicitado, que exige uma tabela `REVIEW` persistida no `hotel.db`. Agora os dados relacionais ficam no SQLite e as consultas de negócio usam essa tabela. Mongita permanece como camada documental complementar, não como substituto do SQLite.


## 3-4. Modelo relacional e restrições

Ative-se a integridade referencial, com PK/FK, campos essenciais `NOT NULL`, unicidade das comodidades, checks de ratings, booleanos, ocupação/proporções, valores não negativos e domínios de ação/outcome. O outcome mantém os literais da fonte (`better`, `worse`, `in line`).

In [5]:
SCHEMA = """
BEGIN;
CREATE TABLE HOTEL(hotel_id TEXT PRIMARY KEY, name TEXT NOT NULL, brand TEXT, tier TEXT NOT NULL, rooms INTEGER NOT NULL CHECK(rooms>=0), opened_year INTEGER, distance_to_water_km REAL CHECK(distance_to_water_km IS NULL OR distance_to_water_km>=0), neighbourhood TEXT, comp_set TEXT, positioning_index REAL, base_rate REAL CHECK(base_rate IS NULL OR base_rate>=0), city TEXT NOT NULL, review_count INTEGER CHECK(review_count IS NULL OR review_count>=0), lifetime_rating REAL CHECK(lifetime_rating IS NULL OR lifetime_rating BETWEEN 1 AND 5), share_rating_5 REAL CHECK(share_rating_5 IS NULL OR share_rating_5 BETWEEN 0 AND 1), share_rating_1_2 REAL CHECK(share_rating_1_2 IS NULL OR share_rating_1_2 BETWEEN 0 AND 1));
CREATE TABLE ROOM_TYPE(room_type_id TEXT PRIMARY KEY, hotel_id TEXT NOT NULL REFERENCES HOTEL, room_type TEXT NOT NULL, rooms INTEGER NOT NULL CHECK(rooms>=0), max_occupancy INTEGER NOT NULL CHECK(max_occupancy>0), size_sqm REAL CHECK(size_sqm IS NULL OR size_sqm>=0), rate_premium_pct REAL, bar_rate REAL CHECK(bar_rate IS NULL OR bar_rate>=0), view TEXT, tier TEXT);
CREATE TABLE AMENITY(amenity_id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE);
CREATE TABLE HOTEL_AMENITY(hotel_id TEXT NOT NULL REFERENCES HOTEL, amenity_id INTEGER NOT NULL REFERENCES AMENITY, PRIMARY KEY(hotel_id,amenity_id));
CREATE TABLE REVIEW_SCENARIO(scenario_id TEXT PRIMARY KEY, label TEXT NOT NULL, share_of_reviews REAL CHECK(share_of_reviews IS NULL OR share_of_reviews BETWEEN 0 AND 1), rating_band TEXT, trigger TEXT, expected_sentiment TEXT, recovery_action TEXT);
CREATE TABLE REVIEW(review_id TEXT PRIMARY KEY, hotel_id TEXT NOT NULL REFERENCES HOTEL, scenario_id TEXT NOT NULL REFERENCES REVIEW_SCENARIO, review_date TEXT NOT NULL, stay_date TEXT NOT NULL, week_start TEXT NOT NULL, rating INTEGER NOT NULL CHECK(rating BETWEEN 1 AND 5), title TEXT, text TEXT, channel TEXT, segment TEXT, nights INTEGER CHECK(nights IS NULL OR nights>=0), room_type TEXT, verified_stay INTEGER NOT NULL CHECK(verified_stay IN (0,1)), cleanliness INTEGER CHECK(cleanliness IS NULL OR cleanliness BETWEEN 1 AND 5), comfort INTEGER CHECK(comfort IS NULL OR comfort BETWEEN 1 AND 5), service_score INTEGER CHECK(service_score IS NULL OR service_score BETWEEN 1 AND 5), location_score INTEGER CHECK(location_score IS NULL OR location_score BETWEEN 1 AND 5), value_score INTEGER CHECK(value_score IS NULL OR value_score BETWEEN 1 AND 5), sleep_quality INTEGER CHECK(sleep_quality IS NULL OR sleep_quality BETWEEN 1 AND 5), would_return INTEGER CHECK(would_return IS NULL OR would_return IN (0,1)), sentiment TEXT, sentiment_score REAL, lexicon_version TEXT, response_text TEXT, response_hours REAL CHECK(response_hours IS NULL OR response_hours>=0));
CREATE TABLE WEEKLY_PERFORMANCE(hotel_id TEXT NOT NULL REFERENCES HOTEL, week_start TEXT NOT NULL, available_rooms INTEGER CHECK(available_rooms IS NULL OR available_rooms>=0), rooms_sold INTEGER CHECK(rooms_sold IS NULL OR rooms_sold>=0), bookings INTEGER CHECK(bookings IS NULL OR bookings>=0), cancellations INTEGER CHECK(cancellations IS NULL OR cancellations>=0), occupancy REAL CHECK(occupancy IS NULL OR occupancy BETWEEN 0 AND 1), adr REAL CHECK(adr IS NULL OR adr>=0), revpar REAL CHECK(revpar IS NULL OR revpar>=0), room_revenue REAL CHECK(room_revenue IS NULL OR room_revenue>=0), ancillary_revenue REAL CHECK(ancillary_revenue IS NULL OR ancillary_revenue>=0), commission REAL CHECK(commission IS NULL OR commission>=0), net_revenue REAL CHECK(net_revenue IS NULL OR net_revenue>=0), total_revenue REAL CHECK(total_revenue IS NULL OR total_revenue>=0), trevpar REAL CHECK(trevpar IS NULL OR trevpar>=0), revpor REAL CHECK(revpor IS NULL OR revpor>=0), net_revpar REAL CHECK(net_revpar IS NULL OR net_revpar>=0), gross_operating_profit REAL, goppar REAL, cpor REAL CHECK(cpor IS NULL OR cpor>=0), length_of_stay REAL CHECK(length_of_stay IS NULL OR length_of_stay>=0), booking_window_days REAL CHECK(booking_window_days IS NULL OR booking_window_days>=0), cancellation_rate REAL CHECK(cancellation_rate IS NULL OR cancellation_rate BETWEEN 0 AND 1), comp_set_occupancy REAL CHECK(comp_set_occupancy IS NULL OR comp_set_occupancy BETWEEN 0 AND 1), comp_set_adr REAL CHECK(comp_set_adr IS NULL OR comp_set_adr>=0), comp_set_revpar REAL CHECK(comp_set_revpar IS NULL OR comp_set_revpar>=0), rgi REAL, ari REAL, mpi REAL, ancillary_capture REAL CHECK(ancillary_capture IS NULL OR ancillary_capture BETWEEN 0 AND 1), demand_event TEXT, event_nights INTEGER CHECK(event_nights IS NULL OR event_nights>=0), flow_through REAL, review_count INTEGER CHECK(review_count IS NULL OR review_count>=0), mean_rating REAL CHECK(mean_rating IS NULL OR mean_rating BETWEEN 1 AND 5), positive_share REAL CHECK(positive_share IS NULL OR positive_share BETWEEN 0 AND 1), sentiment_index REAL, response_rate REAL CHECK(response_rate IS NULL OR response_rate BETWEEN 0 AND 1), PRIMARY KEY(hotel_id,week_start));
CREATE TABLE DAILY_MARKET(hotel_id TEXT NOT NULL REFERENCES HOTEL, date TEXT NOT NULL, weekday TEXT, month INTEGER CHECK(month IS NULL OR month BETWEEN 1 AND 12), year INTEGER, season TEXT NOT NULL, demand_event TEXT, rooms_available INTEGER CHECK(rooms_available IS NULL OR rooms_available>=0), rooms_sold INTEGER CHECK(rooms_sold IS NULL OR rooms_sold>=0), occupancy REAL NOT NULL CHECK(occupancy BETWEEN 0 AND 1), adr REAL CHECK(adr IS NULL OR adr>=0), room_revenue REAL CHECK(room_revenue IS NULL OR room_revenue>=0), ancillary_revenue REAL CHECK(ancillary_revenue IS NULL OR ancillary_revenue>=0), commission REAL CHECK(commission IS NULL OR commission>=0), net_revenue REAL CHECK(net_revenue IS NULL OR net_revenue>=0), gross_operating_profit REAL, bookings INTEGER CHECK(bookings IS NULL OR bookings>=0), cancellations INTEGER CHECK(cancellations IS NULL OR cancellations>=0), cancellation_rate REAL CHECK(cancellation_rate IS NULL OR cancellation_rate BETWEEN 0 AND 1), booking_window_days REAL CHECK(booking_window_days IS NULL OR booking_window_days>=0), length_of_stay REAL CHECK(length_of_stay IS NULL OR length_of_stay>=0), available_rooms INTEGER CHECK(available_rooms IS NULL OR available_rooms>=0), revpar REAL CHECK(revpar IS NULL OR revpar>=0), trevpar REAL CHECK(trevpar IS NULL OR trevpar>=0), revpor REAL CHECK(revpor IS NULL OR revpor>=0), net_revpar REAL CHECK(net_revpar IS NULL OR net_revpar>=0), goppar REAL, cpor REAL CHECK(cpor IS NULL OR cpor>=0), ancillary_capture REAL CHECK(ancillary_capture IS NULL OR ancillary_capture BETWEEN 0 AND 1), market_occupancy REAL CHECK(market_occupancy IS NULL OR market_occupancy BETWEEN 0 AND 1), market_adr REAL CHECK(market_adr IS NULL OR market_adr>=0), demand_index REAL, otb_occupancy_index REAL, comp_set_occupancy REAL CHECK(comp_set_occupancy IS NULL OR comp_set_occupancy BETWEEN 0 AND 1), comp_set_adr REAL CHECK(comp_set_adr IS NULL OR comp_set_adr>=0), comp_set_revpar REAL CHECK(comp_set_revpar IS NULL OR comp_set_revpar>=0), rgi REAL, ari REAL, mpi REAL, mars_rate REAL CHECK(mars_rate IS NULL OR mars_rate>=0), mars_band_low REAL CHECK(mars_band_low IS NULL OR mars_band_low>=0), mars_band_high REAL CHECK(mars_band_high IS NULL OR mars_band_high>=0), rate_position REAL, rate_gap_pct REAL, rate_action TEXT, suggested_rate REAL CHECK(suggested_rate IS NULL OR suggested_rate>=0), suggested_change_pct REAL, rate_guardrail TEXT, rate_floor REAL CHECK(rate_floor IS NULL OR rate_floor>=0), rate_ceiling REAL CHECK(rate_ceiling IS NULL OR rate_ceiling>=0), price_elasticity REAL, mars_basis TEXT, mars_confidence REAL CHECK(mars_confidence IS NULL OR mars_confidence BETWEEN 0 AND 1), PRIMARY KEY(hotel_id,date));
CREATE TABLE RATE_DECISION(decision_id TEXT PRIMARY KEY, hotel_id TEXT NOT NULL REFERENCES HOTEL, week_start TEXT NOT NULL, action TEXT NOT NULL CHECK(action IN ('raise','hold','lower')), current_adr REAL CHECK(current_adr IS NULL OR current_adr>=0), mars_rate REAL CHECK(mars_rate IS NULL OR mars_rate>=0), rate_position REAL, rate_gap_pct REAL, booking_pace_index REAL, forecast_occupancy REAL CHECK(forecast_occupancy IS NULL OR forecast_occupancy BETWEEN 0 AND 1), suggested_rate REAL CHECK(suggested_rate IS NULL OR suggested_rate>=0), suggested_change_pct REAL, rate_guardrail TEXT, expected_revpar REAL CHECK(expected_revpar IS NULL OR expected_revpar>=0), expected_lift_pct REAL, confidence REAL CHECK(confidence IS NULL OR confidence BETWEEN 0 AND 1), rationale TEXT, cancellation_rate REAL CHECK(cancellation_rate IS NULL OR cancellation_rate BETWEEN 0 AND 1), demand_event TEXT, realised_occupancy REAL CHECK(realised_occupancy IS NULL OR realised_occupancy BETWEEN 0 AND 1), realised_adr REAL CHECK(realised_adr IS NULL OR realised_adr>=0), realised_revpar REAL CHECK(realised_revpar IS NULL OR realised_revpar>=0), realised_uplift_pct REAL, outcome TEXT NOT NULL CHECK(outcome IN ('better','worse','in line')), FOREIGN KEY(hotel_id,week_start) REFERENCES WEEKLY_PERFORMANCE(hotel_id,week_start));
CREATE INDEX idx_room_hotel ON ROOM_TYPE(hotel_id);
CREATE INDEX idx_weekly_week ON WEEKLY_PERFORMANCE(week_start);
CREATE INDEX idx_daily_date ON DAILY_MARKET(date);
CREATE INDEX idx_daily_season ON DAILY_MARKET(season);
CREATE INDEX idx_rate_hotel_week ON RATE_DECISION(hotel_id,week_start);
CREATE INDEX idx_rate_action ON RATE_DECISION(action);
CREATE INDEX idx_review_hotel ON REVIEW(hotel_id);
CREATE INDEX idx_review_scenario ON REVIEW(scenario_id);
CREATE INDEX idx_review_rating ON REVIEW(rating);
CREATE INDEX idx_review_week ON REVIEW(week_start);
"""

## 5. Carga dos dados

O arquivo do banco é recriado em cada execução. A carga usa uma transação; listas de comodidades são desmembradas e fatos não recebem novamente nome, tier ou conjunto competitivo. Datas ficam no formato ISO em `TEXT` e booleanos viram `0/1`.

In [6]:
def value(v):
    if isinstance(v,bool): return int(v)
    if isinstance(v,str) and v=="": return None
    return v
def insert(conn, table, columns, rows):
    names=", ".join(f'"{c}"' for c in columns)
    marks=", ".join("?" for _ in columns)
    conn.executemany(f"INSERT INTO {table} ({names}) VALUES ({marks})", [tuple(value(r.get(c)) for c in columns) for r in rows])
def cols(rows, excluded=()):
    return [c for c in rows[0] if c not in excluded]

if DB.exists(): DB.unlink()
conn=sqlite3.connect(DB)
try:
    conn.execute("PRAGMA foreign_keys=ON")
    with conn:
        conn.executescript(SCHEMA)
        insert(conn,"HOTEL",cols(data["hotels.jsonl"],["amenities"]),data["hotels.jsonl"])
        amenity_names=sorted({a for h in data["hotels.jsonl"] for a in h["amenities"]})
        conn.executemany("INSERT INTO AMENITY(name) VALUES (?)",[(a,) for a in amenity_names])
        amenity_ids=dict(conn.execute("SELECT name,amenity_id FROM AMENITY"))
        conn.executemany("INSERT INTO HOTEL_AMENITY VALUES (?,?)",[(h["hotel_id"],amenity_ids[a]) for h in data["hotels.jsonl"] for a in h["amenities"]])
        for filename,table,excluded in [("room_types.jsonl","ROOM_TYPE",()),("review_scenarios.jsonl","REVIEW_SCENARIO",()),("reviews.jsonl","REVIEW",()),("weekly_performance.jsonl","WEEKLY_PERFORMANCE",("hotel_name","tier","comp_set")),("daily_market.jsonl","DAILY_MARKET",("hotel_name","tier","comp_set")),("rate_decisions.jsonl","RATE_DECISION",())]:
            insert(conn,table,cols(data[filename],excluded),data[filename])
    MONGITA= BANK / "hotel_mongita.db"
    if MONGITA.is_dir(): shutil.rmtree(MONGITA)
    elif MONGITA.exists(): MONGITA.unlink()
    mongo_client=MongitaClientDisk(str(MONGITA))
    collection=mongo_client.hotel.reviews_document
    collection.create_index("review_id")
    documents=[{"_id":row["review_id"],**{k:value(v) for k,v in row.items()}} for row in data["reviews.jsonl"]]
    if documents: collection.insert_many(documents)
    assert collection.count_documents({})==len(documents)
    print("SQLite recriado:",DB)
    print("Mongita recriado:",MONGITA,"| documentos:",collection.count_documents({}))
finally:
    conn.close()
print("Conexão SQLite encerrada.")

SQLite recriado: /home/japa/Documentos/entrega_final_ciencia_de_dados/banco/hotel.db
Mongita recriado: /home/japa/Documentos/entrega_final_ciencia_de_dados/banco/hotel_mongita.db | documentos: 12077
Conexão SQLite encerrada.


## 6. Verificação da integridade referencial

In [7]:
conn = sqlite3.connect(DB)
try:
    conn.execute("PRAGMA foreign_keys=ON")
    orphan_sql = {
        "reviews sem hotel": "SELECT r.review_id FROM REVIEW r LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "reviews sem cenário": "SELECT r.review_id FROM REVIEW r LEFT JOIN REVIEW_SCENARIO s USING(scenario_id) WHERE s.scenario_id IS NULL",
        "decisões sem hotel": "SELECT d.decision_id FROM RATE_DECISION d LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "decisões sem semana correspondente": "SELECT d.decision_id FROM RATE_DECISION d LEFT JOIN WEEKLY_PERFORMANCE w ON w.hotel_id=d.hotel_id AND w.week_start=d.week_start WHERE w.hotel_id IS NULL",
        "registros diarios sem hotel": "SELECT d.hotel_id,d.date FROM DAILY_MARKET d LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "tipos de quarto sem hotel": "SELECT r.room_type_id FROM ROOM_TYPE r LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "semanas sem hotel": "SELECT w.hotel_id,w.week_start FROM WEEKLY_PERFORMANCE w LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "hotel_amenity sem hotel": "SELECT ha.hotel_id FROM HOTEL_AMENITY ha LEFT JOIN HOTEL h USING(hotel_id) WHERE h.hotel_id IS NULL",
        "hotel_amenity sem amenity": "SELECT ha.amenity_id FROM HOTEL_AMENITY ha LEFT JOIN AMENITY a USING(amenity_id) WHERE a.amenity_id IS NULL",
    }
    orphan_counts=[{"verificação":label,"órfãos":len(conn.execute(sql).fetchall())} for label,sql in orphan_sql.items()]
    display(pd.DataFrame(orphan_counts))
    fk=conn.execute("PRAGMA foreign_key_check").fetchall()
    print("PRAGMA foreign_key_check:", fk)
    assert all(row["órfãos"]==0 for row in orphan_counts)
    assert not fk
finally:
    conn.close()


,verificação,órfãos
0,reviews sem hotel,0
1,reviews sem cenário,0
2,decisões sem hotel,0
3,decisões sem semana correspondente,0
4,registros diarios sem hotel,0
5,tipos de quarto sem hotel,0
6,semanas sem hotel,0
7,hotel_amenity sem hotel,0
8,hotel_amenity sem amenity,0


PRAGMA foreign_key_check: []


## 7. Consultas de negócio

Todas as respostas desta seção são calculadas por consultas SQL contra o SQLite persistente. A tabela `REVIEW` foi adicionada ao `hotel.db` para que avaliações, hotéis e cenários participem das junções relacionais e das verificações de integridade. Mongita continua guardando uma cópia documental completa das avaliações, mas não substitui nem monopoliza o modelo relacional: é uma camada complementar para leitura flexível de payloads textuais.

Em `RATE_DECISION`, `raise` significa aumentar tarifa, `hold` manter e `lower` reduzir. Os resultados são associações observadas, não evidência causal.


In [8]:
conn = sqlite3.connect(DB)
queries={
"7.1 Receita por hotel/ano (top 5)":"""WITH totals AS (SELECT CAST(substr(w.week_start,1,4) AS INTEGER) year,h.hotel_id,h.name hotel_name,SUM(w.total_revenue) total_revenue FROM WEEKLY_PERFORMANCE w JOIN HOTEL h USING(hotel_id) GROUP BY year,h.hotel_id,h.name), ranked AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY year ORDER BY total_revenue DESC) rank FROM totals) SELECT year,rank,hotel_id,hotel_name,ROUND(total_revenue,2) total_revenue FROM ranked WHERE rank<=5 ORDER BY year,rank""",
"7.2 Ocupação e ADR por tier e estação":"""SELECT h.tier,d.season,ROUND(AVG(d.occupancy)*100,2) ocupacao_media_pct,ROUND(AVG(d.adr),2) adr_medio FROM DAILY_MARKET d JOIN HOTEL h USING(hotel_id) GROUP BY h.tier,d.season ORDER BY h.tier,d.season""",
"7.3 Hotéis com mais de 30% de notas 1-2":"""SELECT h.hotel_id,h.name hotel_name,COUNT(*) total_avaliacoes,SUM(r.rating IN (1,2)) avaliacoes_1_2,ROUND(1.0*SUM(r.rating IN (1,2))/COUNT(*),4) proporcao,ROUND(100.0*SUM(r.rating IN (1,2))/COUNT(*),2) percentual FROM REVIEW r JOIN HOTEL h USING(hotel_id) GROUP BY h.hotel_id,h.name HAVING 1.0*SUM(r.rating IN (1,2))/COUNT(*)>0.30 ORDER BY proporcao DESC""",
"7.4 Cenários de avaliação":"""SELECT s.scenario_id,s.label,COUNT(r.review_id) quantidade_avaliacoes,ROUND(AVG(r.rating),2) nota_media,s.recovery_action FROM REVIEW r JOIN REVIEW_SCENARIO s USING(scenario_id) GROUP BY s.scenario_id,s.label,s.recovery_action ORDER BY quantidade_avaliacoes DESC""",
"7.5 Decisões por ação e outcome better":"""SELECT action,COUNT(*) total_decisoes,SUM(outcome='better') quantidade_better,ROUND(1.0*SUM(outcome='better')/COUNT(*),4) proporcao_better,ROUND(100.0*SUM(outcome='better')/COUNT(*),2) percentual_better FROM RATE_DECISION GROUP BY action ORDER BY CASE action WHEN 'raise' THEN 1 WHEN 'hold' THEN 2 ELSE 3 END""",
"7.6 Respostas por nota":"""SELECT rating,COUNT(*) total_avaliacoes,SUM(response_text IS NOT NULL AND TRIM(response_text)!='') quantidade_respondida,ROUND(100.0*SUM(response_text IS NOT NULL AND TRIM(response_text)!='')/COUNT(*),2) percentual_respondida,ROUND(AVG(CASE WHEN response_text IS NOT NULL AND TRIM(response_text)!='' THEN response_hours END),2) tempo_medio_resposta_horas FROM REVIEW GROUP BY rating ORDER BY rating""",
"7.7 Em hotéis luxury, quais cenários têm ao menos 50 avaliações e como são nota e resposta?":"""SELECT s.scenario_id,s.label,COUNT(r.review_id) quantidade_avaliacoes,ROUND(AVG(r.rating),2) nota_media,ROUND(AVG(CASE WHEN TRIM(COALESCE(r.response_text,''))!='' THEN r.response_hours END),2) tempo_medio_resposta_horas,ROUND(100.0*SUM(TRIM(COALESCE(r.response_text,''))!='')/COUNT(*),2) percentual_respondida FROM REVIEW r JOIN HOTEL h USING(hotel_id) JOIN REVIEW_SCENARIO s USING(scenario_id) WHERE h.tier='luxury' GROUP BY s.scenario_id,s.label HAVING COUNT(r.review_id)>=50 ORDER BY quantidade_avaliacoes DESC"""
}
results={}
try:
    for title,sql in queries.items():
        print(title,"\nSQL:",sql)
        results[title]=pd.read_sql_query(sql,conn)
        display(results[title])
finally:
    conn.close()


7.1 Receita por hotel/ano (top 5) 
SQL: WITH totals AS (SELECT CAST(substr(w.week_start,1,4) AS INTEGER) year,h.hotel_id,h.name hotel_name,SUM(w.total_revenue) total_revenue FROM WEEKLY_PERFORMANCE w JOIN HOTEL h USING(hotel_id) GROUP BY year,h.hotel_id,h.name), ranked AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY year ORDER BY total_revenue DESC) rank FROM totals) SELECT year,rank,hotel_id,hotel_name,ROUND(total_revenue,2) total_revenue FROM ranked WHERE rank<=5 ORDER BY year,rank


,year,rank,hotel_id,hotel_name,total_revenue
0,2027,1,H03,Marisol Bay Resort,26247578.01
1,2027,2,H06,Seabright Grand,16222286.70
2,2027,3,H01,Alder & Quay Hotel,12361889.51
3,2027,4,H04,Harborlight Suites,11051825.01
4,2027,5,H05,The Copper Meridian,10663427.13
5,2028,1,H03,Marisol Bay Resort,26254391.07
6,2028,2,H06,Seabright Grand,16137153.75
7,2028,3,H01,Alder & Quay Hotel,12415432.15
8,2028,4,H04,Harborlight Suites,11066841.80
9,2028,5,H05,The Copper Meridian,10619552.67


7.2 Ocupação e ADR por tier e estação 
SQL: SELECT h.tier,d.season,ROUND(AVG(d.occupancy)*100,2) ocupacao_media_pct,ROUND(AVG(d.adr),2) adr_medio FROM DAILY_MARKET d JOIN HOTEL h USING(hotel_id) GROUP BY h.tier,d.season ORDER BY h.tier,d.season


,tier,season,ocupacao_media_pct,adr_medio
0,boutique,autumn,74.97,205.37
1,boutique,spring,74.97,204.03
2,boutique,summer,86.78,253.95
3,boutique,winter,66.17,179.76
4,economy,autumn,60.97,88.68
5,economy,spring,61.23,88.31
6,economy,summer,71.48,108.97
7,economy,winter,53.82,78.46
8,luxury,autumn,71.48,357.39
9,luxury,spring,71.46,355.56


7.3 Hotéis com mais de 30% de notas 1-2 
SQL: SELECT h.hotel_id,h.name hotel_name,COUNT(*) total_avaliacoes,SUM(r.rating IN (1,2)) avaliacoes_1_2,ROUND(1.0*SUM(r.rating IN (1,2))/COUNT(*),4) proporcao,ROUND(100.0*SUM(r.rating IN (1,2))/COUNT(*),2) percentual FROM REVIEW r JOIN HOTEL h USING(hotel_id) GROUP BY h.hotel_id,h.name HAVING 1.0*SUM(r.rating IN (1,2))/COUNT(*)>0.30 ORDER BY proporcao DESC


,hotel_id,hotel_name,total_avaliacoes,avaliacoes_1_2,proporcao,percentual
0,H08,Northgate Inn,718,281,0.3914,39.14
1,H12,Ridgeline Motor Inn,506,184,0.3636,36.36
2,H07,The Wren Rooms,289,103,0.3564,35.64
3,H10,Kestrel Park Hotel,1169,411,0.3516,35.16
4,H14,Willowmere Lodge,679,221,0.3255,32.55
5,H09,Cedar & Salt Hotel,963,304,0.3157,31.57
6,H13,Paloma Court Hotel,829,251,0.3028,30.28


... (4 outputs left out)

7.6 Respostas por nota 
SQL: SELECT rating,COUNT(*) total_avaliacoes,SUM(response_text IS NOT NULL AND TRIM(response_text)!='') quantidade_respondida,ROUND(100.0*SUM(response_text IS NOT NULL AND TRIM(response_text)!='')/COUNT(*),2) percentual_respondida,ROUND(AVG(CASE WHEN response_text IS NOT NULL AND TRIM(response_text)!='' THEN response_hours END),2) tempo_medio_resposta_horas FROM REVIEW GROUP BY rating ORDER BY rating


,rating,total_avaliacoes,quantidade_respondida,percentual_respondida,tempo_medio_resposta_horas
0,1,1087,1087,100.00,21.21
1,2,2439,2439,100.00,21.18
2,3,3212,3212,100.00,21.06
3,4,2302,663,28.80,20.19
4,5,3037,907,29.86,21.38


7.7 Em hotéis luxury, quais cenários têm ao menos 50 avaliações e como são nota e resposta? 
SQL: SELECT s.scenario_id,s.label,COUNT(r.review_id) quantidade_avaliacoes,ROUND(AVG(r.rating),2) nota_media,ROUND(AVG(CASE WHEN TRIM(COALESCE(r.response_text,''))!='' THEN r.response_hours END),2) tempo_medio_resposta_horas,ROUND(100.0*SUM(TRIM(COALESCE(r.response_text,''))!='')/COUNT(*),2) percentual_respondida FROM REVIEW r JOIN HOTEL h USING(hotel_id) JOIN REVIEW_SCENARIO s USING(scenario_id) WHERE h.tier='luxury' GROUP BY s.scenario_id,s.label HAVING COUNT(r.review_id)>=50 ORDER BY quantidade_avaliacoes DESC


,scenario_id,label,quantidade_avaliacoes,nota_media,tempo_medio_resposta_horas,percentual_respondida
0,delight,Stay exceeded expectations,408,4.88,22.39,29.90
1,location,Location and access,267,4.62,17.48,28.84
2,service_recovery,Service recovery,254,4.44,19.79,28.35
3,room_condition,Room condition,231,2.65,20.68,100.00
4,noise,Noise and sleep,214,2.51,22.05,100.00
5,checkin_wait,Check-in and queue,175,2.96,20.04,84.00
6,breakfast,Breakfast and food,172,3.87,20.05,51.16
7,billing,Billing and charges,167,1.79,20.42,100.00
8,cleanliness,Cleanliness,134,1.88,19.71,100.00
9,wifi,Wifi and connectivity,125,2.90,21.27,84.80


### Interpretação dos resultados

- **7.1 — Receita:** Marisol Bay Resort (H03) lidera nos dois anos: 26.247.578,01 em 2027 e 26.254.391,07 em 2028. O ranking soma os fatos semanais de receita total.
- **7.2 — Tier e estação:** o verão apresenta a maior ocupação média e o inverno a menor em todos os tiers. Luxury tem o maior ADR médio em cada estação; economy, o menor. A sazonalidade aparece tanto na ocupação quanto no preço.
- **7.3 — Notas baixas:** sete hotéis ultrapassam 30% de avaliações com 1 ou 2 estrelas. Northgate Inn (H08) tem a maior proporção (39,14%); Paloma Court Hotel (H13) fica mais próximo do limite (30,28%). São candidatos prioritários para investigar as causas das avaliações.
- **7.4 — Cenários:** “Stay exceeded expectations” (delight) é o cenário mais frequente (1.840 avaliações) e tem média 4,84. Billing (1,63) e cleanliness (1,69) têm as menores médias; as ações recomendadas de auditoria/refundo e reinspeção/limpeza correspondem a problemas operacionais concretos.
- **7.5 — Tarifas:** outcome `better` ocorre em 42,86% das reduções (81/189), 38,33% dos aumentos (243/634) e 37,64% das manutenções (233/619). A diferença é uma associação observada, não evidência de que reduzir preço causou melhores resultados.
- **7.6 — Respostas:** 100% das notas 1–3 receberam resposta, contra 28,80% das notas 4 e 29,86% das notas 5. Entre as respostas, o tempo médio varia de 20,19 h (nota 4) a 21,38 h (nota 5). A cobertura menor nas avaliações positivas pode orientar uma política de resposta mais abrangente.
- **7.7 — Luxury:** delight concentra 408 avaliações (média 4,88), mas somente 29,90% foram respondidas. Room condition e noise tiveram notas médias 2,65 e 2,51; billing e cleanliness, 1,79 e 1,88. Embora estes quatro cenários tenham 100% de respostas no recorte, suas notas sugerem que responder precisa vir acompanhado de ações corretivas.

As avaliações usadas nesses cálculos vieram de Mongita. Para viabilizar `GROUP BY` e os `JOIN`s exigidos, foram analisadas em uma base SQLite `:memory:` descartável; o banco persistente mantém REVIEWS apenas no Mongita.

## 8. Conclusão e principais achados

As tabelas e achados apresentados a seguir refletem as consultas SQL executadas acima. Interprete associações de preço como descritivas, não causais. Limitações: os resultados cobrem apenas o período e os campos disponíveis nos JSONL; `city` foi mantida em HOTEL por não haver dimensão urbana detalhada. Próximos passos: revisar a política de atualização dos snapshots e ampliar os dados temporais/geográficos antes de inferências.

In [9]:
conn=sqlite3.connect(DB)
try:
    counts=pd.read_sql_query("""SELECT 'HOTEL' tabela,COUNT(*) registros FROM HOTEL UNION ALL SELECT 'ROOM_TYPE',COUNT(*) FROM ROOM_TYPE UNION ALL SELECT 'AMENITY',COUNT(*) FROM AMENITY UNION ALL SELECT 'HOTEL_AMENITY',COUNT(*) FROM HOTEL_AMENITY UNION ALL SELECT 'REVIEW_SCENARIO',COUNT(*) FROM REVIEW_SCENARIO UNION ALL SELECT 'REVIEW',COUNT(*) FROM REVIEW UNION ALL SELECT 'WEEKLY_PERFORMANCE',COUNT(*) FROM WEEKLY_PERFORMANCE UNION ALL SELECT 'DAILY_MARKET',COUNT(*) FROM DAILY_MARKET UNION ALL SELECT 'RATE_DECISION',COUNT(*) FROM RATE_DECISION""",conn)
    display(counts)
    print("Avaliações persistidas no SQLite:",conn.execute("SELECT COUNT(*) FROM REVIEW").fetchone()[0])
    print("Avaliações persistidas no Mongita:",collection.count_documents({}))
    print("Ranking top 5 por ano:")
    display(results["7.1 Receita por hotel/ano (top 5)"])
finally:
    conn.close()


,tabela,registros
0,HOTEL,14
1,ROOM_TYPE,42
2,AMENITY,33
3,HOTEL_AMENITY,57
4,REVIEW_SCENARIO,11
5,REVIEW,12077
6,WEEKLY_PERFORMANCE,1456
7,DAILY_MARKET,10192
8,RATE_DECISION,1442


Avaliações persistidas no SQLite: 12077
Avaliações persistidas no Mongita: 12077
Ranking top 5 por ano:


,year,rank,hotel_id,hotel_name,total_revenue
0,2027,1,H03,Marisol Bay Resort,26247578.01
1,2027,2,H06,Seabright Grand,16222286.70
2,2027,3,H01,Alder & Quay Hotel,12361889.51
3,2027,4,H04,Harborlight Suites,11051825.01
4,2027,5,H05,The Copper Meridian,10663427.13
5,2028,1,H03,Marisol Bay Resort,26254391.07
6,2028,2,H06,Seabright Grand,16137153.75
7,2028,3,H01,Alder & Quay Hotel,12415432.15
8,2028,4,H04,Harborlight Suites,11066841.80
9,2028,5,H05,The Copper Meridian,10619552.67


## 9. Mongita: avaliações documentais complementares

Os documentos completos permanecem em `banco/hotel_mongita.db`, com `review_id` como identificador único. Além disso, o SQLite relacional agora possui uma tabela `REVIEW` com chaves estrangeiras para `HOTEL` e `REVIEW_SCENARIO`; ela é usada pelas consultas de negócio da Etapa 7.

Mongita é vantajosa para textos longos, campos opcionais, documentos heterogêneos e leitura integral sem joins. O Mongo não substitui o SQLite neste projeto: o SQLite oferece integridade referencial, chaves, restrições e consultas analíticas; Mongita oferece uma representação documental complementar.


In [11]:
client=MongitaClientDisk(str(MONGITA))
collection=client.hotel.reviews_document
review_count=collection.count_documents({})
review_ids=[doc["review_id"] for doc in collection.find({})]
assert review_count==len(review_ids)==len(set(review_ids))
conn=sqlite3.connect(DB)
try:
    sqlite_review_count=conn.execute("SELECT COUNT(*) FROM REVIEW").fetchone()[0]
    sqlite_review_ids=conn.execute("SELECT COUNT(DISTINCT review_id) FROM REVIEW").fetchone()[0]
    assert sqlite_review_count==review_count
    assert sqlite_review_ids==review_count
    assert not conn.execute("PRAGMA foreign_key_check").fetchall()
finally:
    conn.close()
print("Documentos com review_id único:",review_count)
print("Tabela REVIEW no SQLite:",sqlite_review_count,"avaliações")


Documentos com review_id único: 12077
Tabela REVIEW no SQLite: 12077 avaliações
